# Normalisation

A receptionist hands you a spreadsheet. "This is how we track patient visits." It is one massive flat table. Every single row repeats the patient's address, their GP's name and address, the department name... you can already see the problems.

This notebook is about **normalisation** -- the process of structuring a relational database so that redundancy is eliminated and data integrity is protected. We will take a genuinely messy flat table, identify exactly what is wrong with it, and decompose it into a clean, normalised schema.

The stakes are real. In healthcare, a letter sent to the wrong GP address could mean a missed cancer diagnosis. Normalisation is not academic tidiness -- it is a safety mechanism.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
%pip install -q jupysql
%load_ext sql

In [ ]:
%sql sqlite:///../data/nhs_trust.sqlite

In [ ]:
import pandas as pd

## The flat table

Let's load the spreadsheet the receptionist gave us and see what we are dealing with.

In [ ]:
df = pd.read_csv("../data/flat_patient_visits.csv")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(10)

Eighteen columns. Every row is a single visit, but every row also carries a complete copy of the patient's details, the GP practice's details, the department, the doctor -- everything.

Let's count how many times patient information is repeated:

In [ ]:
patient_visits = df.groupby("patient_id").size().reset_index(name="visit_count")
print(f"Unique patients: {patient_visits.shape[0]}")
print(f"Patients with multiple visits: {(patient_visits['visit_count'] > 1).sum()}")
print(f"Max visits for one patient: {patient_visits['visit_count'].max()}")
patient_visits.sort_values("visit_count", ascending=False).head()

Every time a patient visits, their name, date of birth, address, and postcode are copied again. That is not just wasteful -- it is dangerous.

## The three anomalies

Redundant data causes three specific problems, and they all show up in this spreadsheet.

### Update anomaly

When the same fact is stored in multiple places, updating it means finding and changing every copy. Miss one, and your data contradicts itself.

Let's check whether GP practice addresses are consistent across all rows:

In [ ]:
gp_addresses = df.groupby("gp_practice_name")["gp_address"].nunique()
inconsistent_gps = gp_addresses[gp_addresses > 1]
print(f"GP practices with inconsistent addresses: {len(inconsistent_gps)}\n")

for gp_name in inconsistent_gps.index:
    addresses = df[df["gp_practice_name"] == gp_name]["gp_address"].unique()
    print(f"{gp_name}:")
    for addr in addresses:
        count = len(df[(df["gp_practice_name"] == gp_name) & (df["gp_address"] == addr)])
        print(f"  '{addr}' ({count} rows)")
    print()

There it is. The same GP practice appears with different addresses depending on which row you look at. Someone updated the address in some rows but not others. Or different staff typed it differently.

Now imagine posting a referral letter to a GP. Which address do you use? If you pick the wrong one, that letter goes nowhere. In healthcare, that could mean a patient misses a follow-up for a suspicious lump.

This is the **update anomaly**: when redundant data gets out of sync.

### Insert anomaly

Suppose a new GP practice opens in the area. We want to record their details. But in this flat table, every row must also have a patient, a visit date, a department, a diagnosis... We cannot record the GP practice without inventing a fake visit.

This is the **insert anomaly**: we cannot add one fact without fabricating unrelated facts.

### Delete anomaly

Suppose we delete all the visits for a particular patient. If that patient was the only one registered at a small GP practice, we have also destroyed all our information about that practice -- its address, phone number, everything.

This is the **delete anomaly**: removing one fact accidentally destroys unrelated facts.

Let's also check for department floor inconsistencies while we're at it:

In [ ]:
dept_floors = df.groupby("department")["department_floor"].nunique()
inconsistent_depts = dept_floors[dept_floors > 1]
print(f"Departments with inconsistent floor numbers: {len(inconsistent_depts)}\n")

for dept_name in inconsistent_depts.index:
    floors = df[df["department"] == dept_name]["department_floor"].unique()
    print(f"{dept_name}: listed on floors {sorted(floors)}")

Departments apparently exist on two different floors simultaneously. Marvellous.

## First Normal Form (1NF)

Normalisation happens in stages. Each stage -- called a **normal form** -- eliminates a specific type of redundancy.

**First Normal Form** has two requirements:

1. **Atomic values**: every cell contains a single value, not a list or a set.
2. **No repeating groups**: you do not have columns like `diagnosis_1`, `diagnosis_2`, `diagnosis_3`.

Our flat table actually passes 1NF already. Each cell has one value. There are no repeating column groups. That is not unusual -- most spreadsheets people hand you are at least in 1NF.

But being in 1NF does not mean the table is well-designed. It just means we have cleared the lowest bar.

## Functional dependencies

Before we can normalise further, we need the concept of a **functional dependency**. This is the single most important idea in normalisation, and it is simpler than it sounds.

A functional dependency means: *if you know the value of column A, you can determine the value of column B*. We write this as:

```
A -> B
```

Read it as "A determines B".

Let's look at our flat table and identify the functional dependencies:

**Patient dependencies:**
```
patient_id -> patient_name, patient_dob, patient_address, patient_postcode, gp_practice_id
```
If you know the patient ID, you know their name, date of birth, and address. One patient, one set of details.

**GP practice dependencies:**
```
gp_practice_id -> gp_practice_name, gp_address, gp_postcode, gp_phone
```
If you know the GP practice ID, you know the practice name, address, and phone. One practice, one address (or there *should* be).

**Department dependencies:**
```
department -> department_floor
```
Each department is on one floor.

**Doctor dependencies:**
```
doctor_name -> doctor_specialty
```
Each doctor has one specialty.

**The visit itself** is identified by a combination of columns -- the composite key `(patient_id, visit_date, department)` determines the diagnosis, treatment, and follow-up date.

The fundamental insight is this: most of these dependencies have nothing to do with each other. The GP's phone number does not depend on which patient visited which department. Yet they are all crammed into the same row.

## Second Normal Form (2NF)

**Second Normal Form** requires that every non-key column depends on the *entire* primary key, not just part of it.

This only matters when the primary key is a composite -- made up of multiple columns. If the key is a single column, 2NF is automatically satisfied.

In our flat table, if we consider the composite key `(patient_id, visit_date, department)` as identifying each visit, then:

- `patient_name` depends only on `patient_id` (partial dependency)
- `department_floor` depends only on `department` (partial dependency)
- `diagnosis` depends on the full composite key (fine)

Those partial dependencies are 2NF violations. The fix: **pull out the partially dependent columns into their own tables**.

We would create:
- A `patients` table (keyed on `patient_id`)
- A `departments` table (keyed on `department`)
- The visit table keeps only columns that depend on the full composite key

## Third Normal Form (3NF)

**Third Normal Form** goes one step further: no non-key column should depend on another non-key column. This is called a **transitive dependency**.

Here is the classic example from our data:

```
patient_id -> gp_practice_id -> gp_practice_name, gp_address, gp_postcode, gp_phone
```

The patient's row contains their GP practice details. But those details do not depend on the patient -- they depend on the GP practice ID, which happens to be stored in the patient row. That is a transitive dependency: patient -> GP practice ID -> GP address.

The fix: pull GP practice details into their own table. The patient table stores only the `gp_practice_id` as a foreign key.

This is exactly the fix for our update anomaly. Once the GP address exists in only one place (the `gp_practices` table), it is impossible for two rows to disagree about it. You update it once, and every patient referencing that practice automatically uses the correct address.

**That is the whole point of normalisation.** Not tidiness for its own sake, but making certain classes of error structurally impossible.

## Decomposing the flat table

Now let's actually do it. We will decompose our flat table into properly normalised tables using SQL.

Our normalised database already exists (the trust's operational system uses it), so let's look at its structure:

In [ ]:
%sqlcmd tables

Let's examine each table's structure:

In [ ]:
%sqlcmd columns --table gp_practices

In [ ]:
%sqlcmd columns --table patients

In [ ]:
%sqlcmd columns --table departments

In [ ]:
%sqlcmd columns --table doctors

In [ ]:
%sqlcmd columns --table visits

Notice the design:

- **gp_practices**: each practice appears exactly once. One row, one address. Update anomaly? Impossible.
- **patients**: stores patient details plus a `gp_practice_id` foreign key. No GP address here -- just a reference.
- **departments**: each department appears once with its floor number.
- **doctors**: each doctor appears once with their specialty and department.
- **visits**: the actual events. Only IDs referencing the other tables, plus the visit-specific data (diagnosis, treatment, follow-up).

Each table stores exactly one kind of thing. Each fact exists in exactly one place.

## Verifying: the update anomaly is gone

Let's prove it. In the normalised database, how many addresses does each GP practice have?

In [ ]:
%%sql
SELECT name, address, postcode
FROM gp_practices
ORDER BY name
LIMIT 10;

Each practice has exactly one row, therefore exactly one address. There is no second copy that could go out of sync.

If Riverside Medical Centre moves to a new building, we run one UPDATE:

```sql
UPDATE gp_practices
SET address = '1 New Building, Lakeside'
WHERE name = 'Riverside Medical Centre';
```

Every patient registered there, every query that joins to `gp_practices` -- they all immediately see the new address. No hunting through rows. No missed updates. No wrong letters.

## Reconstructing the flat view with JOINs

A common worry: "But what if I need all the data in one view?" You can always reconstruct the flat table using JOINs. You lose nothing by normalising -- you just store the data more intelligently.

In [ ]:
%%sql
SELECT
    p.patient_id,
    p.name AS patient_name,
    p.dob AS patient_dob,
    p.address AS patient_address,
    gp.name AS gp_practice_name,
    gp.address AS gp_address,
    d.name AS department,
    d.floor AS department_floor,
    doc.name AS doctor_name,
    doc.specialty AS doctor_specialty,
    v.visit_date,
    v.diagnosis,
    v.treatment
FROM visits v
JOIN patients p ON v.patient_id = p.patient_id
JOIN gp_practices gp ON p.gp_practice_id = gp.gp_practice_id
JOIN departments d ON v.department_id = d.department_id
JOIN doctors doc ON v.doctor_id = doc.doctor_id
LIMIT 10;

Same information, but assembled on demand from normalised sources. Each fact lives in one place and is never duplicated.

## The relationship between keys

Let's take a moment to look at how the tables are connected.

```
gp_practices  <--  patients  <--  visits  -->  doctors
                                    |
                                    v
                                departments
```

- Each **patient** belongs to one **GP practice** (via `gp_practice_id`)
- Each **visit** references one **patient**, one **doctor**, and one **department**
- Each **doctor** belongs to one **department**

These references are **foreign keys**. A foreign key is a column whose values must match a primary key in another table. It is the mechanism that holds normalised tables together.

For example, `patients.gp_practice_id` is a foreign key referencing `gp_practices.gp_practice_id`. You cannot insert a patient with `gp_practice_id = 999` if no such practice exists -- the database will reject it.

## A practical example: querying the normalised schema

How many patients are registered at each GP practice, and which practices have the most visits?

In [ ]:
%%sql
SELECT
    gp.name AS gp_practice,
    COUNT(DISTINCT p.patient_id) AS registered_patients,
    COUNT(v.visit_id) AS total_visits
FROM gp_practices gp
LEFT JOIN patients p ON gp.gp_practice_id = p.gp_practice_id
LEFT JOIN visits v ON p.patient_id = v.patient_id
GROUP BY gp.name
ORDER BY total_visits DESC
LIMIT 10;

This query would be just as easy with the flat table -- but with normalisation, we know the GP addresses are consistent, and we know adding a new practice does not require fabricating a visit.

## Summary

| Normal Form | Rule | What it prevents |
|---|---|---|
| **1NF** | Atomic values, no repeating groups | Ambiguous cell contents |
| **2NF** | No partial dependencies on composite keys | Redundancy from shared key segments |
| **3NF** | No transitive dependencies | Redundancy from non-key columns depending on other non-key columns |

The practical upshot:

- Each entity (patient, GP practice, department, doctor, visit) gets its own table
- Each fact is stored once
- Tables are linked by foreign keys
- The three anomalies become structurally impossible

Normalisation is not about following rules for the sake of it. It is about designing data structures where entire categories of error simply cannot occur.

---

## Exercises

### Exercise 1: Identify the functional dependencies

Consider a flat table tracking **hospital equipment** with these columns:

```
equipment_id, equipment_name, department_id, department_name, 
department_floor, manufacturer, manufacturer_phone, 
purchase_date, warranty_expiry
```

List all the functional dependencies you can identify. Which are partial dependencies? Which are transitive?

Write your answer as comments in the cell below.

In [ ]:
# Functional dependencies:
#
# YOUR ANSWER HERE
#

### Exercise 2: Find inconsistencies in the flat table

Using pandas, find all patients who appear with different addresses across different rows in the flat CSV. (Hint: group by `patient_id` and check how many unique `patient_address` values each has.)

Why might this happen in practice?

In [ ]:
df = pd.read_csv("../data/flat_patient_visits.csv")

# YOUR CODE HERE


### Exercise 3: Normalise an equipment tracking table

Using the functional dependencies you identified in Exercise 1, write SQL `CREATE TABLE` statements to decompose the equipment table into 3NF.

Your tables should:
- Have appropriate primary keys
- Use foreign keys to link related tables
- Eliminate all partial and transitive dependencies

In [ ]:
%%sql

-- YOUR SQL HERE
-- Create normalised tables for equipment tracking


### Exercise 4: Query the normalised database

Write a SQL query to answer: **which department has the highest number of unique diagnoses?**

Use the normalised `visits`, `departments`, and any other tables you need.

In [ ]:
%%sql

-- YOUR SQL HERE


### Exercise 5: The insert anomaly in practice

The trust has just hired a new doctor, Dr. Sarah Mitchell, who specialises in Endocrinology. She has been assigned to the General Medicine department (department_id = 4) but has not seen any patients yet.

1. Explain why you **cannot** record Dr. Mitchell in the flat table without inventing a fake visit.
2. Write an INSERT statement to add her to the normalised `doctors` table.
3. Write a query to confirm she appears in the database, even though she has zero visits.

In [ ]:
%%sql

-- YOUR SQL HERE
